In [0]:
dbutils.widgets.text('p_batch_id', '')
v_batch_id = dbutils.widgets.get('p_batch_id')

In [0]:
%run ../common/environmental_config

In [0]:
%run ../common/silver-helper

In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"
bronze_table1 = f"{catalog_name}.{bronze_schema}.races"
silver_table1 = f"{catalog_name}.{silver_schema}.races"

In [0]:
from pyspark.sql import functions as F
df_circuits = (
    spark.read.table(bronze_table).filter(F.col("batch_id") == v_batch_id))

In [0]:
#dropping the unwanted columns
df_circuits_selected = df_circuits.drop("url")

In [0]:
circuts_renamed_df = (
    df_circuits_selected
      .withColumnsRenamed({"circuitId" : "circuit_id",
                           "circuitName" : "circuit_name",
                           "lat" : "latitude",
                           "long" : "longitude"}
))

In [0]:
df_circuits_vaild = circuts_renamed_df.filter("circuit_id is not null")

In [0]:
df_circuits_distinct = df_circuits_vaild.dropDuplicates(['circuit_id'])

In [0]:
df_circuits_final = (
    df_circuits_distinct
      .withColumn('circuit_name' , F.initcap(F.col('circuit_name')))
      .withColumn('locality' , F.initcap(F.col('locality')))
      .withColumn('country' , F.initcap(F.col('country')))
      .withColumn('circuit_id' , F.initcap(F.col('circuit_id')))
)

In [0]:
write_to_silver(input_df=df_circuits_final,
                target_table=silver_table,
                merge_condition="t.circuit_id = s.circuit_id",
                columns_to_update=[
                    'circuit_name',
                    'latitude',
                    'longitude',
                    'locality',
                    'country',
                    'ingestion_timestamp',
                    'source_file',
                    'batch_id'
                ]
)

In [0]:
from pyspark.sql import functions as F
df_races = (spark.read.table(bronze_table1).filter(F.col("batch_id") == v_batch_id))
df_races_selected = df_races.drop("url")
races_renamed_df = (
    df_races_selected
      .withColumnsRenamed({"circuitId" : "circuit_id",
                           "raceName" : "race_name",
                           "date" : "race_date"}
))
df_races_distinct = races_renamed_df.dropDuplicates(['season' , 'round'])
df_races_final = (
    df_races_distinct
      .withColumn('circuit_id' , F.initcap(F.col('circuit_id')))
      .withColumn('race_name' , F.initcap(F.col('race_name')))
)
write_to_silver(input_df=df_races_final,
                target_table=silver_table1,
                merge_condition="t.season = s.season AND t.round = s.round",
                columns_to_update=['race_name',
                                   'race_date',
                                   'circuit_id',
                                   'ingestion_timestamp',
                                   'source_file',
                                   'batch_id']
)